In [ ]:
import sys
sys.path.append('../..')  # preprocessing-mitgcm/, so the library packages can be imported

In [ ]:
from rasterstats import zonal_stats
import geopandas as gpd

import os
import numpy as np
import matplotlib.pyplot as plt

from lib.config import ConfigObject

In [ ]:
config = ConfigObject('../../config.json')

In [ ]:
path_grid = config.Paths.GridFolderPath #output

In [ ]:
# to get the grid shape
with open(os.path.join(path_grid, f'x_epsg{config.EpsgProjection}_grid.npy'), 'rb') as f:
    x = np.load(f)

# 1st method: interpolating measured bathymetry on grid

## Load bathymetry asc files

See at the end of the notebook if merging of different tiles is necessary

In [ ]:
raw_bathy_path = config.Paths.RawBathyPath #input

## Load MITgcm grid

In [ ]:
grid = gpd.read_file(os.path.join(path_grid, f"grid_epsg{config.EpsgProjection}.gpkg"))

## Compute average depth per cell

In [ ]:
# Compute zonal statistics
stats = zonal_stats(grid, raw_bathy_path, stats="mean", nodata=config.NoDataRawBathy)

In [ ]:
grid["avg_altitude"] = [s["mean"] for s in stats]

In [ ]:
grid["avg_altitude"].max()

In [ ]:
grid["avg_altitude"].min()

In [ ]:
grid["avg_depth"] = grid["avg_altitude"] - config.LakeAltitude

In [ ]:
grid.loc[grid["avg_depth"] >= -1, "avg_depth"] = 2

grid.loc[
    (grid["avg_depth"] > -3) & (grid["avg_depth"] < -1),
    "avg_depth"
] = -3

In [ ]:
config.LakeAltitude

In [ ]:
grid["avg_altitude"].head()

In [ ]:
grid["avg_depth"] = grid["avg_depth"].fillna(2)
grid.loc[(grid["avg_depth"] > -0.5) & (grid["avg_depth"] < 0), "avg_depth"] = -0.5
depth_array = grid["avg_depth"].to_numpy().reshape(x.shape)

In [ ]:
plt.imshow(depth_array, vmin=-3)
plt.gca().invert_yaxis()
cbar = plt.colorbar(orientation="horizontal")
plt.title("Depth (m)")

# 2nd method: Loading bathymetry from gpkg file (in case it was modified in QGIS)

In [ ]:
import geopandas as gpd

import os
import numpy as np
import matplotlib.pyplot as plt

from lib.config import ConfigObject

In [ ]:
config = ConfigObject('../../config.json')

In [ ]:
path_grid = config.Paths.GridFolderPath #output

In [ ]:
# to get the grid shape
with open(os.path.join(path_grid, f'x_epsg{config.EpsgProjection}_grid.npy'), 'rb') as f:
    x = np.load(f)

In [ ]:
corrected_grid_name = f"grid_with_depth_epsg2056"
grid = gpd.read_file(os.path.join(path_grid, f"{corrected_grid_name}.gpkg"))

In [ ]:
grid["x"] = grid.geometry.centroid.x
grid["y"] = grid.geometry.centroid.y

In [ ]:
from lib.grid_and_bathy import convert_point_coord_to_mitgcm_coord
x_mitgcm, y_mitgcm = convert_point_coord_to_mitgcm_coord(grid["x"], grid["y"], config.EpsgProjection, config)

In [ ]:
grid["x_mitgcm"] = x_mitgcm
grid["y_mitgcm"] = y_mitgcm

In [ ]:
grid["col"] = ((grid["x_mitgcm"] - config.GridResolution/2) / config.GridResolution).round().astype(int)
grid["row"] = ((grid["y_mitgcm"] - config.GridResolution/2) / config.GridResolution).round().astype(int)

In [ ]:
depth_array = np.full(x.shape, np.nan)

depth_array[
    grid["row"].to_numpy(),
    grid["col"].to_numpy()
] = grid["avg_depth"].to_numpy()

In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(depth_array, vmax=3)
plt.gca().invert_yaxis()
cbar = plt.colorbar(orientation="horizontal")
plt.title("Depth (m)")

In [ ]:
fig = plt.figure(figsize=(15,8))
plt.imshow(depth_array, vmin=0, vmax=10)
plt.gca().invert_yaxis()
plt.colorbar()
plt.show()

## Save bathymetry

In [ ]:
config.Paths.GridFolderPath

In [ ]:
# Write gpkg file
grid.to_file(os.path.join(config.Paths.GridFolderPath, f"grid_with_depth_epsg{config.EpsgProjection}.gpkg"), driver="GPKG")

In [ ]:
# Write binary file
fid = open(os.path.join(config.Paths.GridFolderPath, "bathy.bin"), 'wb')
depth_array.astype('>f8').tofile(fid)
fid.close()